In [1]:
import pandas as pd
from google.colab import files

# Upload Excel database
uploaded = files.upload()

# Identify uploaded workbook
file_path = next(
    name for name in uploaded
    if name.endswith(".xlsx")
)

# Load database tables
projects = pd.read_excel(
    file_path,
    sheet_name="Project_Registry"
)

observations = pd.read_excel(
    file_path,
    sheet_name="Project_Annual_Observations"
)

sources = pd.read_excel(
    file_path,
    sheet_name="Source_Register"
)

print("Database loaded successfully!")
print("Projects:", len(projects))
print("Observations:", len(observations))
print("Sources:", len(sources))

Saving Zimbabwe_Lithium_Research_Database_Populated.xlsx to Zimbabwe_Lithium_Research_Database_Populated.xlsx
Database loaded successfully!
Projects: 6
Observations: 39
Sources: 32


In [2]:
# Select our four core lithium projects

core_projects = projects[
    projects["Core_Case_Study"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("yes")
].copy()

core_data = observations[
    observations["Project_ID"].isin(
        core_projects["Project_ID"]
    )
].copy()

# Show all existing indicators
print("EXISTING RESEARCH INDICATORS")
print("=" * 45)

for indicator in sorted(core_data["Indicator"].dropna().unique()):
    print(indicator)

print("\nTotal core observations:", len(core_data))

EXISTING RESEARCH INDICATORS
Acquisition_Price_Paid
Capex_Committed
Capex_Planned_Announced
Capex_Spent_Expansion
Capex_Spent_Reported
Capex_Spent_Sulphate_Plant
Concentrate_Exports
Concentrate_Production_FY2025
Concentrate_Production_FY2026_Forecast
Cumulative_Export_Proceeds
Employment_Direct
Employment_Direct_Local
Lithium_Sulphate_Capacity_Alt
Lithium_Sulphate_Capacity_SMM
Lithium_Sulphate_Exports_Apr_Sep
Ore_Mined_FY2025
Ore_Mined_FY2026_Forecast
Ore_Processing_Capacity
Ore_Reserve_JORC
Petalite_Concentrate_Capacity
Power_Generation_Capacity
Spodumene_Concentrate_Capacity
Tax_Contribution_Q3

Total core observations: 31


In [3]:
# Step 3: Extract production and export evidence

keywords = (
    "Production|Exports|Capacity|Ore_Mined"
)

production_export = core_data[
    core_data["Indicator"].str.contains(
        keywords,
        case=False,
        na=False
    )
].copy()

# Add project names
production_export = production_export.merge(
    projects[["Project_ID", "Project_Name"]],
    on="Project_ID",
    how="left"
)

# Display observations
display(
    production_export[
        [
            "Observation_ID",
            "Project_Name",
            "Year",
            "Indicator",
            "Value",
            "Unit",
            "Actual_or_Forecast",
            "Verification_Status"
        ]
    ]
)

print(
    "Total production, processing and export observations:",
    len(production_export)
)

,Observation_ID,Project_Name,Year,Indicator,Value,Unit,Actual_or_Forecast,Verification_Status
0,OBS_004,Arcadia-Huayou Cobalt,2024,Concentrate_Exports,400000.0,tonnes,Actual,Single-source claim
1,OBS_007,Arcadia-Huayou Cobalt,2025,Lithium_Sulphate_Capacity_SMM,50000.0,tonnes/year,Actual,Corroborated
2,OBS_007B,Arcadia-Huayou Cobalt,2025,Lithium_Sulphate_Capacity_Alt,60000.0,tonnes/year,Planned,Single-source claim
3,OBS_007C,Arcadia-Huayou Cobalt,2026,Lithium_Sulphate_Exports_Apr_Sep,33000.0,tonnes,Actual,Single-source claim
4,OBS_010B,Bikita Minerals,2023,Ore_Processing_Capacity,4000000.0,tonnes/year,Actual,Corroborated
5,OBS_010C,Bikita Minerals,2023,Spodumene_Concentrate_Capacity,300000.0,tonnes/year,Actual,Corroborated
6,OBS_010D,Bikita Minerals,2023,Petalite_Concentrate_Capacity,300000.0,tonnes/year,Actual,Corroborated
7,OBS_011,Bikita Minerals,2025,Concentrate_Exports,300000.0,tonnes,Actual,Single-source claim
8,OBS_016,Sabi Star Lithium-Tantalum Mine,2023,Ore_Processing_Capacity,900000.0,tonnes/year,Actual,Corroborated
9,OBS_018,Sabi Star Lithium-Tantalum Mine,2024,Power_Generation_Capacity,15.0,MW,Actual,Corroborated


Total production, processing and export observations: 14


In [4]:
# Step 4: Classify available evidence

def classify_indicator(indicator):
    indicator = str(indicator).lower()

    if "export" in indicator:
        return "Exports"
    elif "capacity" in indicator:
        return "Capacity"
    elif "production" in indicator or "ore_mined" in indicator:
        return "Production"
    else:
        return "Other"

production_export["Category"] = (
    production_export["Indicator"].apply(classify_indicator)
)

# Count observations by project and category
coverage = pd.crosstab(
    production_export["Project_Name"],
    production_export["Category"]
)

print("PRODUCTION AND EXPORT EVIDENCE COVERAGE")
display(coverage)

PRODUCTION AND EXPORT EVIDENCE COVERAGE


Category,Capacity,Exports,Production
Project_Name,,,
Arcadia-Huayou Cobalt,2,2,0
Bikita Minerals,3,1,0
Sabi Star Lithium-Tantalum Mine,2,0,0
Sandawana Lithium Mine,0,0,4


In [5]:
# Step 5: Examine evidence by reporting status

status_summary = pd.crosstab(
    production_export["Project_Name"],
    production_export["Actual_or_Forecast"]
)

print("ACTUAL VS FORECAST/PLANNED EVIDENCE")
display(status_summary)

print("\nEVIDENCE REQUIRING CAUTION")

caution_data = production_export[
    production_export["Actual_or_Forecast"]
    .astype(str)
    .str.lower()
    .isin(["forecast", "planned"])
]

display(
    caution_data[
        [
            "Project_Name",
            "Year",
            "Indicator",
            "Value",
            "Unit",
            "Actual_or_Forecast"
        ]
    ]
)

ACTUAL VS FORECAST/PLANNED EVIDENCE


Actual_or_Forecast,Actual,Forecast,Planned
Project_Name,,,
Arcadia-Huayou Cobalt,3,0,1
Bikita Minerals,4,0,0
Sabi Star Lithium-Tantalum Mine,2,0,0
Sandawana Lithium Mine,2,2,0



EVIDENCE REQUIRING CAUTION


,Project_Name,Year,Indicator,Value,Unit,Actual_or_Forecast
2,Arcadia-Huayou Cobalt,2025,Lithium_Sulphate_Capacity_Alt,60000.0,tonnes/year,Planned
12,Sandawana Lithium Mine,2026,Ore_Mined_FY2026_Forecast,1800000.0,tonnes,Forecast
13,Sandawana Lithium Mine,2026,Concentrate_Production_FY2026_Forecast,162000.0,tonnes,Forecast


In [6]:
# Step 6: Separate reported actual observations

actual_data = production_export[
    production_export["Actual_or_Forecast"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("actual")
].copy()

# Exclude electricity generation capacity
# from our lithium production/processing comparison
actual_data = actual_data[
    ~actual_data["Indicator"].eq("Power_Generation_Capacity")
]

print("REPORTED ACTUAL LITHIUM OBSERVATIONS")
display(
    actual_data[
        [
            "Project_Name",
            "Year",
            "Indicator",
            "Value",
            "Unit",
            "Verification_Status"
        ]
    ]
)

print("\nTotal observations:", len(actual_data))

REPORTED ACTUAL LITHIUM OBSERVATIONS


,Project_Name,Year,Indicator,Value,Unit,Verification_Status
0,Arcadia-Huayou Cobalt,2024,Concentrate_Exports,400000.0,tonnes,Single-source claim
1,Arcadia-Huayou Cobalt,2025,Lithium_Sulphate_Capacity_SMM,50000.0,tonnes/year,Corroborated
3,Arcadia-Huayou Cobalt,2026,Lithium_Sulphate_Exports_Apr_Sep,33000.0,tonnes,Single-source claim
4,Bikita Minerals,2023,Ore_Processing_Capacity,4000000.0,tonnes/year,Corroborated
5,Bikita Minerals,2023,Spodumene_Concentrate_Capacity,300000.0,tonnes/year,Corroborated
6,Bikita Minerals,2023,Petalite_Concentrate_Capacity,300000.0,tonnes/year,Corroborated
7,Bikita Minerals,2025,Concentrate_Exports,300000.0,tonnes,Single-source claim
8,Sabi Star Lithium-Tantalum Mine,2023,Ore_Processing_Capacity,900000.0,tonnes/year,Corroborated
10,Sandawana Lithium Mine,2025,Ore_Mined_FY2025,343000.0,tonnes,Single-source claim
11,Sandawana Lithium Mine,2025,Concentrate_Production_FY2025,73000.0,tonnes,Single-source claim



Total observations: 10


In [7]:
# Step 7: Classify lithium value-chain activities

def value_chain_stage(indicator):
    indicator = str(indicator).lower()

    if "sulphate" in indicator:
        return "3. Chemical processing"

    elif "concentrate" in indicator:
        return "2. Concentration"

    elif "ore" in indicator:
        return "1. Mining / ore processing"

    else:
        return "Other"

actual_data["Value_Chain_Stage"] = (
    actual_data["Indicator"].apply(value_chain_stage)
)

display(
    actual_data[
        [
            "Project_Name",
            "Indicator",
            "Value_Chain_Stage"
        ]
    ]
)

,Project_Name,Indicator,Value_Chain_Stage
0,Arcadia-Huayou Cobalt,Concentrate_Exports,2. Concentration
1,Arcadia-Huayou Cobalt,Lithium_Sulphate_Capacity_SMM,3. Chemical processing
3,Arcadia-Huayou Cobalt,Lithium_Sulphate_Exports_Apr_Sep,3. Chemical processing
4,Bikita Minerals,Ore_Processing_Capacity,1. Mining / ore processing
5,Bikita Minerals,Spodumene_Concentrate_Capacity,2. Concentration
6,Bikita Minerals,Petalite_Concentrate_Capacity,2. Concentration
7,Bikita Minerals,Concentrate_Exports,2. Concentration
8,Sabi Star Lithium-Tantalum Mine,Ore_Processing_Capacity,1. Mining / ore processing
10,Sandawana Lithium Mine,Ore_Mined_FY2025,1. Mining / ore processing
11,Sandawana Lithium Mine,Concentrate_Production_FY2025,2. Concentration


In [8]:
# Step 8: Create a value-chain evidence matrix

value_chain_matrix = pd.crosstab(
    actual_data["Project_Name"],
    actual_data["Value_Chain_Stage"]
)

# Convert counts into Yes/No
value_chain_matrix = value_chain_matrix.gt(0)

value_chain_matrix = value_chain_matrix.replace({
    True: "Yes",
    False: "No"
})

print("LITHIUM VALUE-CHAIN EVIDENCE MATRIX")
display(value_chain_matrix)

LITHIUM VALUE-CHAIN EVIDENCE MATRIX


Value_Chain_Stage,1. Mining / ore processing,2. Concentration,3. Chemical processing
Project_Name,,,
Arcadia-Huayou Cobalt,No,Yes,Yes
Bikita Minerals,Yes,Yes,No
Sabi Star Lithium-Tantalum Mine,Yes,No,No
Sandawana Lithium Mine,Yes,Yes,No


In [9]:
# Step 9: Classify the type of evidence

def evidence_type(indicator):
    indicator = str(indicator).lower()

    if "exports" in indicator:
        return "Reported exports"
    elif "capacity" in indicator:
        return "Installed/reported capacity"
    elif "production" in indicator or "ore_mined" in indicator:
        return "Reported production"
    else:
        return "Other"

actual_data["Evidence_Type"] = (
    actual_data["Indicator"].apply(evidence_type)
)

# Create a project-level evidence matrix
evidence_matrix = pd.crosstab(
    actual_data["Project_Name"],
    actual_data["Evidence_Type"]
)

print("PROJECT-LEVEL EVIDENCE TYPE MATRIX")
display(evidence_matrix)


PROJECT-LEVEL EVIDENCE TYPE MATRIX


Evidence_Type,Installed/reported capacity,Reported exports,Reported production
Project_Name,,,
Arcadia-Huayou Cobalt,1,2,0
Bikita Minerals,3,1,0
Sabi Star Lithium-Tantalum Mine,1,0,0
Sandawana Lithium Mine,0,0,2


In [10]:
# Step 10: Create comparative research table

comparison = (
    actual_data.groupby(
        ["Project_Name", "Evidence_Type"]
    )
    .size()
    .unstack(fill_value=0)
)

comparison = comparison.reindex(
    columns=[
        "Installed/reported capacity",
        "Reported production",
        "Reported exports"
    ],
    fill_value=0
)

comparison.columns = [
    "Capacity_Observations",
    "Production_Observations",
    "Export_Observations"
]

comparison["Total_Evidence"] = comparison.sum(axis=1)

print("ZIMBABWE LITHIUM COMPARATIVE EVIDENCE TABLE")
display(comparison)

ZIMBABWE LITHIUM COMPARATIVE EVIDENCE TABLE


,Capacity_Observations,Production_Observations,Export_Observations,Total_Evidence
Project_Name,,,,
Arcadia-Huayou Cobalt,1,0,2,3
Bikita Minerals,3,0,1,4
Sabi Star Lithium-Tantalum Mine,1,0,0,1
Sandawana Lithium Mine,0,2,0,2


In [11]:
from google.colab import files

# Save comparative table
comparison.to_csv(
    "Zimbabwe_Lithium_Production_Export_Comparison.csv"
)

# Download CSV
files.download(
    "Zimbabwe_Lithium_Production_Export_Comparison.csv"
)

print("Research comparison table exported successfully!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Research comparison table exported successfully!


### Preliminary Findings: Production, Processing and Exports

This analysis examines production, processing capacity and export evidence from four lithium mining projects in Zimbabwe: Arcadia, Bikita, Sabi Star and Sandawana.

The dataset contains 14 relevant observations, of which 11 are classified as actual, two as forecasts and one as planned. After excluding electricity-generation capacity, 10 reported actual lithium observations remain.

The findings indicate differences in the available evidence across the four projects. Arcadia has reported evidence of concentrate exports and lithium sulphate processing capacity and exports. Bikita has evidence of ore-processing capacity, concentrate-production capacity and concentrate exports. Sabi Star has evidence of ore-processing capacity, while Sandawana has reported ore-mining and concentrate-production figures.

These differences suggest that the four projects can be examined at different points in the lithium value chain. However, the available observations do not establish which project generates the greatest domestic economic value.

**Research limitations:** The observations cover different years and indicators. Capacity figures cannot be treated as actual production, and reported figures require verification against original sources. Missing observations should not be interpreted as evidence that an activity does not exist.

**Next research priority:** Verify the original sources and collect comparable evidence on actual output, export products, domestic processing, employment and economic benefits.